### Rain-on-snow project <br>
#### USGS streamflow time series retrieval

This notebook retrieves NWM retrospective streamflow and USGS observed streaflow for GAGES II (reference and non-reference basins).<br>
Streamflow time series are retrieved using USGS Water dat API, Dataretrieval package. See documentation in: <br>
https://doi-usgs.github.io/dataretrieval-python/index.html <br>
https://github.com/DOI-USGS/dataretrieval-python/tree/main <br>

To have access to higher rate limits, an API Key can be requested at:<br>
https://api.waterdata.usgs.gov/signup/

Once time series are retrieved, several performance metrics will be computed, including:
- NSE
- KGE
- PBias
- Pearson Correlation

In [1]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
import numpy as np
from dataretrieval import waterdata
import functions.rain_on_snow_fncns as ros

1. User specific definitions

In [4]:
######### local paths
# base directory where input metadata is located
base_dir = Path('./input')

# output directory for retrieved data
output_dir = Path('./output')

# location list (USGS gauge IDs)
locations_path = Path(base_dir, 'Metadata_GAGESII_NE_ROS.parquet')

# unique ID column header in the locations list
unique_location_id = 'GAGE_ID'

2. Read locations and retrieve streamflow time series

In [5]:
# create a list of usgs IDs in the format "usgs-xxxxxxxx"
df_ids = pd.read_parquet(locations_path, engine='pyarrow')
id_list = df_ids[unique_location_id].radd('USGS-').to_list()
print(f'Total of GAGES II sites: {len(id_list)}')

Total of GAGES II sites: 913


In [6]:
# Setting API Key as environment variable to allow higher rate limits
os.environ["API_USGS_PAT"] = "QurcJrqNEtcAdf4hrp63e0MYe1ZrjDCaqFodaMud"

In [7]:
# Example for retrieving few sites
siteIDs = id_list[0:2]
parameterCode = "00060"  # Discharge
startDate = "2021-09-01" #"1979-10-01" #
endDate = "2021-09-30" #"2022-09-30" #

# Get the data
discharge = waterdata.get_continuous(
    monitoring_location_id=siteIDs, parameter_code=parameterCode, time=f"{startDate}/{endDate}"
)
print("Retrieved " + str(len(discharge[0])) + " data values.")

Retrieving: continuous · 1 page · 5,569 rows · 996/1,000 requests remaining


Retrieved 5569 data values.


In [8]:
# Display the data frame as a table
display(discharge[0])

,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,continuous_id
0,abd7a6b631874668b953e88cbbf4e374,USGS-01011000,00060,00011,2021-09-01 00:00:00+00:00,369.0,ft^3/s,Approved,None,2025-08-23 14:37:56.810047+00:00,b5b9a4bc-4058-4ac8-a1c1-94c2ba21b953
1,cdaf3cebfd0e49ebbe16231e6cd947e0,USGS-01013500,00060,00011,2021-09-01 00:00:00+00:00,56.3,ft^3/s,Approved,None,2025-08-23 16:25:18.643754+00:00,ab515ca1-ecdb-4670-af3b-60d6012e0c64
2,abd7a6b631874668b953e88cbbf4e374,USGS-01011000,00060,00011,2021-09-01 00:15:00+00:00,369.0,ft^3/s,Approved,None,2025-08-23 14:37:56.810047+00:00,4d20f631-3c92-4182-baec-10ffba526c69
3,cdaf3cebfd0e49ebbe16231e6cd947e0,USGS-01013500,00060,00011,2021-09-01 00:15:00+00:00,56.3,ft^3/s,Approved,None,2025-08-23 16:25:18.643754+00:00,f1594f3f-d84f-43c8-8437-ba3c03f5ae35
4,abd7a6b631874668b953e88cbbf4e374,USGS-01011000,00060,00011,2021-09-01 00:30:00+00:00,369.0,ft^3/s,Approved,None,2025-08-23 14:37:56.810047+00:00,9aef394b-1eb3-4ea0-8e70-4ee4fec4c8f1
...,...,...,...,...,...,...,...,...,...,...,...
5564,cdaf3cebfd0e49ebbe16231e6cd947e0,USGS-01013500,00060,00011,2021-09-29 23:30:00+00:00,524.0,ft^3/s,Approved,None,2025-08-23 16:25:18.643754+00:00,5e183aaa-9340-4c96-92e6-7707c729f603
5565,abd7a6b631874668b953e88cbbf4e374,USGS-01011000,00060,00011,2021-09-29 23:45:00+00:00,743.0,ft^3/s,Approved,None,2025-08-23 16:04:25.394318+00:00,42b4607f-a3ee-4c90-adc3-9ef0d47a1233
5566,cdaf3cebfd0e49ebbe16231e6cd947e0,USGS-01013500,00060,00011,2021-09-29 23:45:00+00:00,524.0,ft^3/s,Approved,None,2025-08-23 16:25:18.643754+00:00,ecc0397b-df5b-4ddb-b85c-8ba296f08847
5567,abd7a6b631874668b953e88cbbf4e374,USGS-01011000,00060,00011,2021-09-30 00:00:00+00:00,743.0,ft^3/s,Approved,None,2025-08-23 16:04:25.394318+00:00,e5ee4435-8d65-4527-b281-a5b3eb7ce10e


In [9]:
print(discharge[0].dtypes)

time_series_id                            str
monitoring_location_id                    str
parameter_code                            str
statistic_id                              str
time                      datetime64[us, UTC]
value                                 float64
unit_of_measure                           str
approval_status                           str
qualifier                              object
last_modified             datetime64[us, UTC]
continuous_id                             str
dtype: object


* Retrieving all sites across the NE

In [6]:
%%time
# Set the parameters for the USGS data retrieval
parameterCode = '00060'  # Discharge
startDate = '1979-10-01'
endDate   = '2022-09-30'
batchSize  = 40    # sites per request; lower to 20-25 if timeouts occur
nWorkers   = 5   # concurrent HTTP requests; lower if the NWIS server returns errors
usgs_output_path = output_dir / 'usgs_streamflow_Q.parquet'

# Intermediate chunk files are saved to output/usgs_streamflow_Q_chunks/ as each
# request completes. If the run is interrupted, re-running this cell will skip
# already-saved chunks and resume from where it left off.
ros.fetch_usgs_data(
    site_list=id_list,
    start_date=startDate,
    end_date=endDate,
    parameter_code=parameterCode,
    batch_size=batchSize,
    max_workers=nWorkers,
    output_path=usgs_output_path
)

Total tasks: 368 | Already done: 368 | Submitting: 0 | Workers: 5

Merging 368 chunk files (streaming, normalized schema)...
Merged → output/usgs_streamflow_Q.parquet (590,356,537 rows (44 empty skipped))
Done! All chunks complete.
CPU times: user 7min 47s, sys: 1min 4s, total: 8min 52s
Wall time: 7min 43s
